In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adbstoragevd001")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/orders.csv"

In [0]:
df_orderss = spark.read.option('header', True)\
                        .option('inferSchema', True)\
                        .csv(ruta)

In [0]:
orders_schema = StructType(fields=[StructField("order_id",              IntegerType(), False),
                                   StructField("user_id",               IntegerType(), False),
                                   StructField("eval_set",              StringType(), True),
                                   StructField("order_number",          IntegerType(), True),
                                   StructField("order_dow",             IntegerType(), True),
                                   StructField("order_hour_of_day",     IntegerType(), True),
                                   StructField("days_since_prior_order",FloatType(), True),
])


In [0]:
df_orders_final = spark.read\
.option('header', True)\
.schema(orders_schema)\
.csv(ruta)

In [0]:
orders_selected_df = df_orders_final.select(col("order_id"), 
                                            col("user_id"),
                                            col("eval_set"),
                                            col("order_number"),
                                            col("order_dow"),
                                            col("order_hour_of_day"),
                                            col("days_since_prior_order")
                                        )

In [0]:
orders_renamed_df = orders_selected_df.withColumnRenamed("order_id", "order_id") \
                                      .withColumnRenamed("user_id", "user_id") \
                                      .withColumnRenamed("eval_set", "eval_set") \
                                      .withColumnRenamed("order_number", "order_number") \
                                      .withColumnRenamed("order_dow", "order_dow") \
                                      .withColumnRenamed("order_hour_of_day", "order_hour_of_day") \
                                      .withColumnRenamed("days_since_prior_order", "days_since_prior_order")

In [0]:
orders_final_df = orders_renamed_df.withColumn("ingestion_date", current_timestamp())

In [0]:
orders_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.bronze_orders")